## Sampling-bias sensitivity of main parameter-change results

This section tests whether the main calibrated-parameter change results are sensitive to the most over- or under-represented CBGs in the SafeGraph home panel.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/frontiers-rev')
print("Current directory files:", os.listdir('.'))

Mounted at /content/drive
Current directory files: ['Manuscript_v1.PDF', 'Reviewer_Comments.pdf', 'Frontiers in Big Data - Reviewer Comments.gdoc', 'figure_table_comparison_top_ses', 'spatial_autocorrelation_check', 'Frontiers_rev_results.gdoc', 'SupMaterial_v1.PDF', 'parameter_values_ses_cluster_comparison_2019.csv', 'nyc_cbgs.json', 'NY_cbg_census.csv', 'PSO_2019_6params_NYC_norm_28_PSO_15.csv', 'PSO_2018_6params_NYC_norm_28_PSO_15.csv', 'PSO_2020_6params_NYC_norm_28_PSO_15.csv', 'PSO_2021_6params_NYC_norm_28_PSO_15.csv', 'cbg_parameter_changes_for_moran.csv', 'moran_global_parameter_changes.csv', 'lisa_local_parameter_changes.csv', 'reviewer_comments_PCA_20260601.ipynb', 'kmeans_stability_nyc_cbgs_results.csv', 'kmeans_stability_cluster_centers.csv', 'kmeans_stability_inertia_table.csv', 'kmeans_stability_ari_table.csv', 'kmeans_stability_cluster_sizes.csv', 'outputs', '03_socioeconomic_spatial_diagnostics.ipynb', '04_model_sensitivity_sampling_bias.ipynb']


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Base directory
# ------------------------------------------------------------

BASE_DIR = Path("/content/drive/MyDrive/frontiers-rev/")
OUTPUT_DIR = BASE_DIR / "outputs_sampling_bias_sensitivity"
OUTPUT_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# PSO parameter files
# ------------------------------------------------------------

PSO_FILES = {
    2018: BASE_DIR / "PSO_2018_6params_NYC_norm_28_PSO_15.csv",
    2019: BASE_DIR / "PSO_2019_6params_NYC_norm_28_PSO_15.csv",
    2020: BASE_DIR / "PSO_2020_6params_NYC_norm_28_PSO_15.csv",
    2021: BASE_DIR / "PSO_2021_6params_NYC_norm_28_PSO_15.csv",
}

# ------------------------------------------------------------
# Representativeness metrics file
# Use the unfiltered CBG-level file if available.
# This file should include CBG ID, census population, SafeGraph devices,
# and/or sampling bias / representation metrics.
# ------------------------------------------------------------

REPRESENTATIVENESS_FILE = BASE_DIR / "nyc_cbg_representativeness_metrics.csv"

# If you want to use the filtered version instead, uncomment this:
# REPRESENTATIVENESS_FILE = BASE_DIR / "nyc_cbg_representativeness_metrics_filtered.csv"

print("Output directory:", OUTPUT_DIR)

for year, path in PSO_FILES.items():
    print(year, path, "exists:", path.exists())

print("Representativeness file exists:", REPRESENTATIVENESS_FILE.exists())

Output directory: /content/drive/MyDrive/frontiers-rev/outputs_sampling_bias_sensitivity
2018 /content/drive/MyDrive/frontiers-rev/PSO_2018_6params_NYC_norm_28_PSO_15.csv exists: True
2019 /content/drive/MyDrive/frontiers-rev/PSO_2019_6params_NYC_norm_28_PSO_15.csv exists: True
2020 /content/drive/MyDrive/frontiers-rev/PSO_2020_6params_NYC_norm_28_PSO_15.csv exists: True
2021 /content/drive/MyDrive/frontiers-rev/PSO_2021_6params_NYC_norm_28_PSO_15.csv exists: True
Representativeness file exists: True


In [3]:
# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def normalize_cbg(series):
    """
    Convert CBG identifiers to 12-digit strings.
    Works for int, float, and string CBG columns.
    """
    return (
        series
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
        .str.zfill(12)
    )


def find_first_existing_column(df, candidates, required=True, label="column"):
    """
    Find the first candidate column that exists in a dataframe.
    """
    for col in candidates:
        if col in df.columns:
            return col

    if required:
        raise ValueError(
            f"Could not find {label}. Tried: {candidates}\n"
            f"Available columns: {df.columns.tolist()}"
        )

    return None


def sign_label(x):
    if pd.isna(x):
        return "missing"
    if x > 0:
        return "positive"
    if x < 0:
        return "negative"
    return "zero"

In [4]:
# ------------------------------------------------------------
# Load NYC CBG representativeness metrics
# ------------------------------------------------------------

rep = pd.read_csv(REPRESENTATIVENESS_FILE)

print("Representativeness file shape:", rep.shape)
print("Representativeness columns:")
print(rep.columns.tolist())
print(rep.head())

# ------------------------------------------------------------
# Detect key columns
# ------------------------------------------------------------

cbg_col_rep = find_first_existing_column(
    rep,
    candidates=[
        "census_block_group",
        "cbg",
        "GEOID",
        "GEOID10",
        "geo_id",
        "geoid",
    ],
    label="CBG ID column in representativeness file"
)

pop_col = find_first_existing_column(
    rep,
    candidates=[
        "census_population",
        "population",
        "populationE",
        "total_population",
        "pop",
    ],
    required=False,
    label="population column"
)

device_col = find_first_existing_column(
    rep,
    candidates=[
        "number_devices_residing",
        "device_count",
        "devices",
        "safegraph_devices",
        "tracked_devices",
        "num_devices",
    ],
    required=False,
    label="device count column"
)

sampling_bias_col = find_first_existing_column(
    rep,
    candidates=[
        "sampling_bias",
        "bias",
        "B",
        "sample_bias",
    ],
    required=False,
    label="sampling bias column"
)

abs_sampling_bias_col = find_first_existing_column(
    rep,
    candidates=[
        "abs_sampling_bias",
        "absolute_sampling_bias",
        "abs_bias",
        "absolute_bias",
        "AB",
    ],
    required=False,
    label="absolute sampling bias column"
)

representation_ratio_col = find_first_existing_column(
    rep,
    candidates=[
        "representation_ratio",
        "rep_ratio",
        "sampling_ratio",
        "sample_ratio",
    ],
    required=False,
    label="representation ratio column"
)

print("\nDetected columns:")
print("CBG column:", cbg_col_rep)
print("Population column:", pop_col)
print("Device column:", device_col)
print("Sampling bias column:", sampling_bias_col)
print("Absolute sampling bias column:", abs_sampling_bias_col)
print("Representation ratio column:", representation_ratio_col)

# ------------------------------------------------------------
# Normalize CBG IDs
# ------------------------------------------------------------

rep["cbg_norm"] = normalize_cbg(rep[cbg_col_rep])

# ------------------------------------------------------------
# Compute sampling bias if not already available
# ------------------------------------------------------------

if abs_sampling_bias_col is not None:
    rep["abs_sampling_bias"] = rep[abs_sampling_bias_col].abs()

    if sampling_bias_col is not None:
        rep["sampling_bias"] = rep[sampling_bias_col]
    else:
        rep["sampling_bias"] = np.nan

elif sampling_bias_col is not None:
    rep["sampling_bias"] = rep[sampling_bias_col]
    rep["abs_sampling_bias"] = rep["sampling_bias"].abs()

else:
    if pop_col is None or device_col is None:
        raise ValueError(
            "The representativeness file does not include sampling bias columns, "
            "and population/device columns could not be detected to compute them."
        )

    rep["device_share"] = rep[device_col] / rep[device_col].sum()
    rep["population_share"] = rep[pop_col] / rep[pop_col].sum()
    rep["sampling_bias"] = rep["device_share"] - rep["population_share"]
    rep["abs_sampling_bias"] = rep["sampling_bias"].abs()

# ------------------------------------------------------------
# Add representation ratio if possible
# ------------------------------------------------------------

if representation_ratio_col is not None:
    rep["representation_ratio_final"] = rep[representation_ratio_col]
elif pop_col is not None and device_col is not None:
    rep["device_share"] = rep[device_col] / rep[device_col].sum()
    rep["population_share"] = rep[pop_col] / rep[pop_col].sum()
    rep["representation_ratio_final"] = rep["device_share"] / rep["population_share"]
else:
    rep["representation_ratio_final"] = np.nan

# ------------------------------------------------------------
# Clean representativeness dataframe
# ------------------------------------------------------------

rep_clean = rep.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["cbg_norm", "abs_sampling_bias"]
).copy()

rep_clean = rep_clean.drop_duplicates(subset=["cbg_norm"])

print("\nRepresentativeness rows after cleaning:", len(rep_clean))
print(rep_clean[["cbg_norm", "sampling_bias", "abs_sampling_bias", "representation_ratio_final"]].head())

print("\nAbsolute sampling bias summary:")
print(rep_clean["abs_sampling_bias"].describe())

Representativeness file shape: (6493, 15)
Representativeness columns:
['census_block_group', 'census_population', 'number_devices_residing', 'state_fips', 'county_fips', 'state', 'county', 'borough', 'bias', 'abs_bias', 'expected_devices_if_representative', 'device_count_gap', 'devices_per_100_residents', 'representation_ratio', 'log_representation_ratio']
   census_block_group  census_population  number_devices_residing  state_fips  \
0        360050001000                  0                      1.0          36   
1        360050001001               7503                     21.0          36   
2        360050002000                  0                      0.0          36   
3        360050002001               2114                     53.0          36   
4        360050002002               2168                     99.0          36   

   county_fips state        county borough      bias  abs_bias  \
0            5    NY  Bronx County   Bronx  0.000183  0.000183   
1            5    NY  

In [5]:
# ------------------------------------------------------------
# Parameter columns from PSO files
# ------------------------------------------------------------

CBG_COL_PSO = "cbg"

PARAMETER_COLS_ORIGINAL = [
    "H_Area_of_store",
    "R_Percentage_of_Visits_by_brand",
    "J_POI_count_where_store_is",
    "K_POI_diversity_where_store_is",
    "L_Demographic_similarity",
    "G_Distance_between_cbg_and_store",
]

PARAMETER_RENAME = {
    "H_Area_of_store": "store_area",
    "R_Percentage_of_Visits_by_brand": "chain_loyalty",
    "J_POI_count_where_store_is": "poi_count",
    "K_POI_diversity_where_store_is": "poi_diversity",
    "L_Demographic_similarity": "demographic_similarity",
    "G_Distance_between_cbg_and_store": "distance",
}

PARAMETER_COLS = list(PARAMETER_RENAME.values())

# ------------------------------------------------------------
# Read PSO files
# ------------------------------------------------------------

params_list = []

for year, path in PSO_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing PSO file for {year}: {path}")

    df_year = pd.read_csv(path)

    print(f"\nYear {year} shape:", df_year.shape)
    print(f"Year {year} columns:", df_year.columns.tolist())

    required_cols = [CBG_COL_PSO] + PARAMETER_COLS_ORIGINAL
    missing_cols = [col for col in required_cols if col not in df_year.columns]

    if missing_cols:
        raise ValueError(
            f"Missing columns in {year} PSO file: {missing_cols}\n"
            f"Available columns: {df_year.columns.tolist()}"
        )

    df_year = df_year[required_cols].copy()
    df_year["year"] = year
    df_year["cbg_norm"] = normalize_cbg(df_year[CBG_COL_PSO])
    df_year = df_year.rename(columns=PARAMETER_RENAME)

    params_list.append(df_year)

params = pd.concat(params_list, ignore_index=True)

print("\nCombined PSO parameter data shape:", params.shape)
print(params.head())

print("\nCBG counts by year:")
print(params.groupby("year")["cbg_norm"].nunique())

# ------------------------------------------------------------
# Keep only CBGs present in all four years
# ------------------------------------------------------------

cbg_year_counts = params.groupby("cbg_norm")["year"].nunique()
complete_cbgs = cbg_year_counts[cbg_year_counts == 4].index

params = params[params["cbg_norm"].isin(complete_cbgs)].copy()

print("\nNumber of CBGs present in all four years:", len(complete_cbgs))
print("PSO parameter rows after keeping complete CBGs:", len(params))


Year 2018 shape: (5502, 8)
Year 2018 columns: ['cbg', 'cost', 'H_Area_of_store', 'R_Percentage_of_Visits_by_brand', 'J_POI_count_where_store_is', 'K_POI_diversity_where_store_is', 'L_Demographic_similarity', 'G_Distance_between_cbg_and_store']

Year 2019 shape: (5502, 8)
Year 2019 columns: ['cbg', 'cost', 'H_Area_of_store', 'R_Percentage_of_Visits_by_brand', 'J_POI_count_where_store_is', 'K_POI_diversity_where_store_is', 'L_Demographic_similarity', 'G_Distance_between_cbg_and_store']

Year 2020 shape: (5502, 8)
Year 2020 columns: ['cbg', 'cost', 'H_Area_of_store', 'R_Percentage_of_Visits_by_brand', 'J_POI_count_where_store_is', 'K_POI_diversity_where_store_is', 'L_Demographic_similarity', 'G_Distance_between_cbg_and_store']

Year 2021 shape: (5502, 8)
Year 2021 columns: ['cbg', 'cost', 'H_Area_of_store', 'R_Percentage_of_Visits_by_brand', 'J_POI_count_where_store_is', 'K_POI_diversity_where_store_is', 'L_Demographic_similarity', 'G_Distance_between_cbg_and_store']

Combined PSO parame

In [6]:
# ------------------------------------------------------------
# Merge PSO parameter data with representativeness metrics
# ------------------------------------------------------------

params_rep = params.merge(
    rep_clean[
        [
            "cbg_norm",
            "sampling_bias",
            "abs_sampling_bias",
            "representation_ratio_final"
        ]
    ],
    on="cbg_norm",
    how="inner"
)

print("Rows after merging PSO parameters with representativeness metrics:", len(params_rep))
print("CBGs after merge:", params_rep["cbg_norm"].nunique())

print("\nCBG counts by year after merge:")
print(params_rep.groupby("year")["cbg_norm"].nunique())

# Check whether any model CBGs were lost in merge
model_cbgs = set(params["cbg_norm"].unique())
merged_cbgs = set(params_rep["cbg_norm"].unique())
lost_cbgs = model_cbgs - merged_cbgs

print("\nModel CBGs:", len(model_cbgs))
print("Merged CBGs:", len(merged_cbgs))
print("Lost CBGs after representativeness merge:", len(lost_cbgs))

if len(lost_cbgs) > 0:
    print("First 10 lost CBGs:", list(lost_cbgs)[:10])

Rows after merging PSO parameters with representativeness metrics: 22008
CBGs after merge: 5502

CBG counts by year after merge:
year
2018    5502
2019    5502
2020    5502
2021    5502
Name: cbg_norm, dtype: int64

Model CBGs: 5502
Merged CBGs: 5502
Lost CBGs after representativeness merge: 0


In [7]:
# ------------------------------------------------------------
# Define sample sets for sensitivity analysis
# ------------------------------------------------------------

rep_analytical = (
    params_rep[
        [
            "cbg_norm",
            "sampling_bias",
            "abs_sampling_bias",
            "representation_ratio_final"
        ]
    ]
    .drop_duplicates(subset=["cbg_norm"])
    .copy()
)

# Rank CBGs by absolute sampling bias within analytical sample
rep_analytical["abs_bias_rank_pct"] = rep_analytical["abs_sampling_bias"].rank(
    method="first",
    pct=True
)

full_cbgs = set(rep_analytical["cbg_norm"])

top_10_abs_bias_cbgs = set(
    rep_analytical.loc[
        rep_analytical["abs_bias_rank_pct"] > 0.90,
        "cbg_norm"
    ]
)

top_20_abs_bias_cbgs = set(
    rep_analytical.loc[
        rep_analytical["abs_bias_rank_pct"] > 0.80,
        "cbg_norm"
    ]
)

sample_sets = {
    "Full analytical sample": full_cbgs,
    "Excluding top 10% absolute sampling-bias CBGs": full_cbgs - top_10_abs_bias_cbgs,
    "Excluding top 20% absolute sampling-bias CBGs": full_cbgs - top_20_abs_bias_cbgs,
}

print("Sample sizes:")
for sample_name, cbg_set in sample_sets.items():
    print(f"{sample_name}: {len(cbg_set)} CBGs")

print("\nAbsolute sampling-bias thresholds:")
print("90th percentile:", rep_analytical["abs_sampling_bias"].quantile(0.90))
print("80th percentile:", rep_analytical["abs_sampling_bias"].quantile(0.80))

rep_analytical.to_csv(
    OUTPUT_DIR / "analytical_cbgs_sampling_bias_metrics.csv",
    index=False
)

Sample sizes:
Full analytical sample: 5502 CBGs
Excluding top 10% absolute sampling-bias CBGs: 4951 CBGs
Excluding top 20% absolute sampling-bias CBGs: 4401 CBGs

Absolute sampling-bias thresholds:
90th percentile: 0.009339239351069983
80th percentile: 0.007194908254785461


In [8]:
# ------------------------------------------------------------
# Helper function: mean parameter values and percent changes
# ------------------------------------------------------------

def compute_parameter_change_summary(df, cbg_set, sample_name):
    sample_df = df[df["cbg_norm"].isin(cbg_set)].copy()

    means = (
        sample_df
        .groupby("year")[PARAMETER_COLS]
        .mean()
        .sort_index()
    )

    required_years = [2018, 2019, 2020, 2021]
    missing_years = [year for year in required_years if year not in means.index]

    if missing_years:
        raise ValueError(f"{sample_name} is missing years: {missing_years}")

    records = []

    for param in PARAMETER_COLS:
        value_2018 = means.loc[2018, param]
        value_2019 = means.loc[2019, param]
        value_2020 = means.loc[2020, param]
        value_2021 = means.loc[2021, param]

        change_2018_2019 = value_2019 - value_2018
        change_2019_2020 = value_2020 - value_2019
        change_2020_2021 = value_2021 - value_2020
        change_2019_2021 = value_2021 - value_2019

        pct_change_2018_2019 = 100 * change_2018_2019 / value_2018 if value_2018 != 0 else np.nan
        pct_change_2019_2020 = 100 * change_2019_2020 / value_2019 if value_2019 != 0 else np.nan
        pct_change_2020_2021 = 100 * change_2020_2021 / value_2020 if value_2020 != 0 else np.nan
        pct_change_2019_2021 = 100 * change_2019_2021 / value_2019 if value_2019 != 0 else np.nan

        records.append({
            "sample": sample_name,
            "n_cbgs": len(cbg_set),
            "parameter": param,
            "mean_2018": value_2018,
            "mean_2019": value_2019,
            "mean_2020": value_2020,
            "mean_2021": value_2021,
            "change_2018_2019": change_2018_2019,
            "change_2019_2020": change_2019_2020,
            "change_2020_2021": change_2020_2021,
            "change_2019_2021": change_2019_2021,
            "pct_change_2018_2019": pct_change_2018_2019,
            "pct_change_2019_2020": pct_change_2019_2020,
            "pct_change_2020_2021": pct_change_2020_2021,
            "pct_change_2019_2021": pct_change_2019_2021,
        })

    return pd.DataFrame(records)


# ------------------------------------------------------------
# Run sensitivity analysis
# ------------------------------------------------------------

sensitivity_results = []

for sample_name, cbg_set in sample_sets.items():
    sensitivity_results.append(
        compute_parameter_change_summary(
            params_rep,
            cbg_set,
            sample_name
        )
    )

sensitivity_results = pd.concat(sensitivity_results, ignore_index=True)

print("Sampling-bias sensitivity results:")
print(sensitivity_results)

sensitivity_results.to_csv(
    OUTPUT_DIR / "sampling_bias_sensitivity_parameter_changes_long.csv",
    index=False
)

Sampling-bias sensitivity results:
                                           sample  n_cbgs  \
0                          Full analytical sample    5502   
1                          Full analytical sample    5502   
2                          Full analytical sample    5502   
3                          Full analytical sample    5502   
4                          Full analytical sample    5502   
5                          Full analytical sample    5502   
6   Excluding top 10% absolute sampling-bias CBGs    4951   
7   Excluding top 10% absolute sampling-bias CBGs    4951   
8   Excluding top 10% absolute sampling-bias CBGs    4951   
9   Excluding top 10% absolute sampling-bias CBGs    4951   
10  Excluding top 10% absolute sampling-bias CBGs    4951   
11  Excluding top 10% absolute sampling-bias CBGs    4951   
12  Excluding top 20% absolute sampling-bias CBGs    4401   
13  Excluding top 20% absolute sampling-bias CBGs    4401   
14  Excluding top 20% absolute sampling-bias CBGs 

In [9]:
# ------------------------------------------------------------
# Compact percent-change tables
# ------------------------------------------------------------

compact_2018_2019 = sensitivity_results.pivot_table(
    index="parameter",
    columns="sample",
    values="pct_change_2018_2019"
)

compact_2019_2020 = sensitivity_results.pivot_table(
    index="parameter",
    columns="sample",
    values="pct_change_2019_2020"
)

compact_2020_2021 = sensitivity_results.pivot_table(
    index="parameter",
    columns="sample",
    values="pct_change_2020_2021"
)

compact_2019_2021 = sensitivity_results.pivot_table(
    index="parameter",
    columns="sample",
    values="pct_change_2019_2021"
)

print("\nPercent change 2018-2019:")
print(compact_2018_2019.round(3))

print("\nPercent change 2019-2020:")
print(compact_2019_2020.round(3))

print("\nPercent change 2020-2021:")
print(compact_2020_2021.round(3))

print("\nPercent change 2019-2021:")
print(compact_2019_2021.round(3))

compact_2018_2019.to_csv(
    OUTPUT_DIR / "sampling_bias_sensitivity_pct_change_2018_2019.csv"
)

compact_2019_2020.to_csv(
    OUTPUT_DIR / "sampling_bias_sensitivity_pct_change_2019_2020.csv"
)

compact_2020_2021.to_csv(
    OUTPUT_DIR / "sampling_bias_sensitivity_pct_change_2020_2021.csv"
)

compact_2019_2021.to_csv(
    OUTPUT_DIR / "sampling_bias_sensitivity_pct_change_2019_2021.csv"
)


Percent change 2018-2019:
sample                  Excluding top 10% absolute sampling-bias CBGs  \
parameter                                                               
chain_loyalty                                                  -0.082   
demographic_similarity                                         -0.610   
distance                                                       -0.434   
poi_count                                                      -0.471   
poi_diversity                                                   1.142   
store_area                                                      1.361   

sample                  Excluding top 20% absolute sampling-bias CBGs  \
parameter                                                               
chain_loyalty                                                  -0.336   
demographic_similarity                                         -0.498   
distance                                                       -0.577   
poi_count              

In [10]:
# ------------------------------------------------------------
# Sign-stability check
# ------------------------------------------------------------

period_cols = [
    "pct_change_2018_2019",
    "pct_change_2019_2020",
    "pct_change_2020_2021",
    "pct_change_2019_2021",
]

sign_stability_records = []

for period_col in period_cols:
    temp = sensitivity_results[["sample", "parameter", period_col]].copy()
    temp["sign"] = temp[period_col].apply(sign_label)

    sign_pivot = temp.pivot_table(
        index="parameter",
        columns="sample",
        values="sign",
        aggfunc="first"
    )

    sign_pivot["period"] = period_col.replace("pct_change_", "")
    sign_pivot = sign_pivot.reset_index()

    sign_stability_records.append(sign_pivot)

sign_stability = pd.concat(sign_stability_records, ignore_index=True)

print("\nSign stability across sampling-bias sensitivity samples:")
print(sign_stability)

sign_stability.to_csv(
    OUTPUT_DIR / "sampling_bias_sensitivity_sign_stability.csv",
    index=False
)


Sign stability across sampling-bias sensitivity samples:
sample               parameter Excluding top 10% absolute sampling-bias CBGs  \
0                chain_loyalty                                      negative   
1       demographic_similarity                                      negative   
2                     distance                                      negative   
3                    poi_count                                      negative   
4                poi_diversity                                      positive   
5                   store_area                                      positive   
6                chain_loyalty                                      positive   
7       demographic_similarity                                      positive   
8                     distance                                      negative   
9                    poi_count                                      positive   
10               poi_diversity                                

In [11]:
# ------------------------------------------------------------
# Magnitude difference from full sample
# This helps quantify whether excluding biased CBGs changes results substantially.
# ------------------------------------------------------------

full_results = sensitivity_results[
    sensitivity_results["sample"] == "Full analytical sample"
].copy()

comparison_samples = [
    "Excluding top 10% absolute sampling-bias CBGs",
    "Excluding top 20% absolute sampling-bias CBGs",
]

difference_records = []

for comparison_sample in comparison_samples:
    comp_results = sensitivity_results[
        sensitivity_results["sample"] == comparison_sample
    ].copy()

    merged = full_results.merge(
        comp_results,
        on="parameter",
        suffixes=("_full", "_comparison")
    )

    for _, row in merged.iterrows():
        for period in [
            "pct_change_2018_2019",
            "pct_change_2019_2020",
            "pct_change_2020_2021",
            "pct_change_2019_2021",
        ]:
            full_value = row[f"{period}_full"]
            comparison_value = row[f"{period}_comparison"]

            difference_records.append({
                "comparison_sample": comparison_sample,
                "parameter": row["parameter"],
                "period": period.replace("pct_change_", ""),
                "full_sample_pct_change": full_value,
                "comparison_sample_pct_change": comparison_value,
                "absolute_difference_percentage_points": comparison_value - full_value,
                "relative_difference_percent": (
                    100 * (comparison_value - full_value) / abs(full_value)
                    if full_value != 0 else np.nan
                )
            })

difference_results = pd.DataFrame(difference_records)

print("\nDifferences from full sample:")
print(difference_results.round(3))

difference_results.to_csv(
    OUTPUT_DIR / "sampling_bias_sensitivity_difference_from_full_sample.csv",
    index=False
)


Differences from full sample:
                                comparison_sample               parameter  \
0   Excluding top 10% absolute sampling-bias CBGs              store_area   
1   Excluding top 10% absolute sampling-bias CBGs              store_area   
2   Excluding top 10% absolute sampling-bias CBGs              store_area   
3   Excluding top 10% absolute sampling-bias CBGs              store_area   
4   Excluding top 10% absolute sampling-bias CBGs           chain_loyalty   
5   Excluding top 10% absolute sampling-bias CBGs           chain_loyalty   
6   Excluding top 10% absolute sampling-bias CBGs           chain_loyalty   
7   Excluding top 10% absolute sampling-bias CBGs           chain_loyalty   
8   Excluding top 10% absolute sampling-bias CBGs               poi_count   
9   Excluding top 10% absolute sampling-bias CBGs               poi_count   
10  Excluding top 10% absolute sampling-bias CBGs               poi_count   
11  Excluding top 10% absolute sampling-bias 

In [15]:
# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("Sampling-bias sensitivity analysis complete.")
print("\nOutputs saved in:", OUTPUT_DIR)

print("\nImportant files:")
print(OUTPUT_DIR / "analytical_cbgs_sampling_bias_metrics.csv")
print(OUTPUT_DIR / "sampling_bias_sensitivity_parameter_changes_long.csv")
print(OUTPUT_DIR / "sampling_bias_sensitivity_pct_change_2019_2020.csv")
print(OUTPUT_DIR / "sampling_bias_sensitivity_pct_change_2020_2021.csv")
print(OUTPUT_DIR / "sampling_bias_sensitivity_pct_change_2019_2021.csv")
print(OUTPUT_DIR / "sampling_bias_sensitivity_sign_stability.csv")
print(OUTPUT_DIR / "sampling_bias_sensitivity_difference_from_full_sample.csv")

print("\nMain table to inspect for reviewer response:")
print(compact_2019_2020)
print(compact_2020_2021)
print(compact_2019_2021)
print(sign_stability)

Sampling-bias sensitivity analysis complete.

Outputs saved in: /content/drive/MyDrive/frontiers-rev/outputs_sampling_bias_sensitivity

Important files:
/content/drive/MyDrive/frontiers-rev/outputs_sampling_bias_sensitivity/analytical_cbgs_sampling_bias_metrics.csv
/content/drive/MyDrive/frontiers-rev/outputs_sampling_bias_sensitivity/sampling_bias_sensitivity_parameter_changes_long.csv
/content/drive/MyDrive/frontiers-rev/outputs_sampling_bias_sensitivity/sampling_bias_sensitivity_pct_change_2019_2020.csv
/content/drive/MyDrive/frontiers-rev/outputs_sampling_bias_sensitivity/sampling_bias_sensitivity_pct_change_2020_2021.csv
/content/drive/MyDrive/frontiers-rev/outputs_sampling_bias_sensitivity/sampling_bias_sensitivity_pct_change_2019_2021.csv
/content/drive/MyDrive/frontiers-rev/outputs_sampling_bias_sensitivity/sampling_bias_sensitivity_sign_stability.csv
/content/drive/MyDrive/frontiers-rev/outputs_sampling_bias_sensitivity/sampling_bias_sensitivity_difference_from_full_sample.csv